# Step 8 — MobileNetV3-Small backbone  (Phase 5, RQ4 prep)  ·  Kaggle

Adds the **second mandatory backbone** from proposal §5A and re-runs the winning
Phase-3 recipe on it, so the thesis can say what the whole framework costs on a
genuinely small CNN.

| | ResNet-18 (Steps 4-7) | MobileNetV3-Small (this step) |
|---|---|---|
| backbone params | ~11.7 M | ~2.5 M |
| pooled feature dim | 512 | **576** |
| in-block adapter sites | `layer1..layer4` last block | `features.{3,6,8,11}` (stage-final residual blocks) |

**What this notebook runs** — 4 configs × (train + 600-episode eval):

| config | adapter | head | result suffix |
|---|---|---|---|
| `exp_phase5_mbnet_bottleneck_evidential` | post-pool Bottleneck-16 (576→16→576) | evidential | `phase5_mbnet` |
| `exp_phase5_mbnet_bottleneck_softmax` | post-pool Bottleneck-16 | softmax | `phase5_mbnet` |
| `exp_phase5_mbnet_parallel_evidential` | **parallel in-block** (Step-6 winner) | evidential | `phase5_mbnet_parallel` |
| `exp_phase5_mbnet_parallel_softmax` | **parallel in-block** | softmax | `phase5_mbnet_parallel` |

Everything except the backbone is held fixed at the Step-4.5/6 recipe (same
adapter form and rank, same cosine prototype metric, same R-EDL loss knobs, same
LR, same episodic schedule, same frozen splits and episode seeds), so the
comparison isolates the **backbone**. The ResNet-18 rows in the final table are
**reused** from Steps 4.5 / 7 — nothing is re-run, so no earlier result changes.

**What you get out** (all under `/kaggle/working/thesis`, plus a single zip at
`/kaggle/working` — see the last section):
* `results/phase5_mbnet*_metrics.json` — 4 result JSONs (600 episodes each)
* `results/phase5_backbone_table.json` — the cross-backbone table
* `results/step8_backbone_comparison.png`, `results/step8_params_vs_accuracy.png`
* `checkpoints/model_phase2_mbnet_{bottleneck,parallel}_prototype-{evidential,softmax}_seed42.pt`
  — the 4 trained adapters (**the models**)

**Kaggle settings:** Accelerator = **GPU T4**, Internet = **ON**,
`+ Add Input → bpeft-data`. Then run top to bottom (~1-2 h; MobileNetV3-Small is
cheap). Expect the run cell to be the long one.

## How we work with the data in this project (read before running)

These are the standing rules for every experiment in this repo. They are what
makes results comparable across runs, steps and machines — breaking one silently
invalidates a comparison rather than crashing, so they matter more than they look.

### 1. In-distribution data: CIFAR-FS with the frozen Bertinetto split
* Source is plain **CIFAR-100** (`data/cifar-100-python.tar.gz`), re-partitioned
  into the canonical **Bertinetto 2019 64 / 16 / 20 class split** — 64 meta-train,
  16 meta-val, 20 meta-test classes, **pairwise disjoint**, union = all 100.
* The split lives in `data/cifar_fs_split.json` and is **FROZEN — never
  regenerate or hand-edit it.** `scripts/build_cifar_fs_split.py` re-materialises
  the same canonical file after a fresh clone (`data/` is gitignored); the loader
  asserts disjointness + full coverage at import and *warns loudly* if it ever
  falls back to the synthetic split. If you see `_status: synthetic_fallback`,
  **stop** — do not report numbers from that run.
* Labels are re-indexed per split (train→0..63, val→0..15, test→0..19), and
  `train`/`val` are drawn from CIFAR-100's train partition, `test` from its test
  partition. Images are resized **32→224** and ImageNet-normalised, because the
  backbones are ImageNet-pretrained.

### 2. Episodes are sampled from seeds, not shuffled
Every episode (5-way, 5-shot support + 15-query) is drawn by a **deterministic
per-episode seed**, so "episode 37" is the same 100 images for everyone:

| stream | seeds | file | rule |
|---|---|---|---|
| **test** | `0..599` | `configs/test_episodes.yaml` | **frozen**; 600 episodes; report on these |
| **val** | `10000..10099` | `configs/val_episodes.yaml` | **frozen**; early stopping + all tuning |
| **train** | `20000+` | `cfg.trainer.train_seed_offset` | new offset each epoch, so epochs don't repeat episodes |

### 3. The one rule that protects the whole thesis: tune on VAL only
Any hyperparameter choice — LR, KL weight, evidence affine, rank, placement,
temperature — is selected on the **val** stream (seeds 10000-10099). The **600
test seeds are touched exactly once, to report.** This is enforced by convention,
not by code, so it is on you. The temperature-scaling baseline follows the same
rule: `evaluate.py` fits `T` on val episodes and applies it unchanged to test.

### 4. OOD pools (evaluated per episode against that episode's prototypes)
| pool | regime | source | n |
|---|---|---|---|
| `svhn_far` | far | SVHN test (`data/svhn/test_32x32.mat`) | 500 |
| `gaussian_far` | far | seeded N(0,1), clamped [-3,3] — sanity check | 500 |
| `cifar100_near` | near | the 16 CIFAR-FS **val** classes (clean, provably disjoint from the 20 test classes) | 500 |
| `tin_near` | near | TinyImageNet train — **uncurated**, may overlap CIFAR-FS semantics | 500 |

The near-OOD claim rests on `cifar100_near`; `tin_near` corroborates it and is
reported with that caveat attached (see `step_writeups/step7.txt`).

### 5. Never extract many-file archives onto a network filesystem
TinyImageNet is a ~120k-file zip. Extracting it onto Google Drive once hung for
**over an hour** (FUSE does a network round-trip per file). The rule, implemented
in `src/datasets/tinyimagenet_ood.py`: cache the **single zip**, copy it once to
local disk, and read the images **straight out of the zip — never extract.** Apply
the same rule to any new heavy dataset.

### 6. Determinism is a result, not a nicety
`set_seed()` seeds python/numpy/torch (CPU+CUDA) and forces
`cudnn.deterministic`; metrics are dumped with `sort_keys=True`. **The same config
run twice must produce a byte-identical `metrics.json`.** If a re-run differs,
something non-deterministic crept in and the numbers cannot be trusted until it
is found. The JSON key schema is likewise frozen: adding a key would make every
earlier step's committed result file differ from its own re-run.

### 7. On Kaggle specifically
`data/` is gitignored, so nothing ships with the clone. Attach the **bpeft-data**
dataset (`cifar-100-python.tar.gz` + `test_32x32.mat`, optionally
`tiny-imagenet-200.zip`) and let the staging cell copy them into `data/`. CIFAR is
MD5-checked. Anything not attached downloads at runtime, which needs Internet ON.

## 0. GPU + environment check

In [ ]:
import torch, sys
print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU T4'

## 1. Clone the repo + install deps

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'                      # <-- branch with the Step 8 commit
REPO_DIR = '/kaggle/working/thesis'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
!pip -q install -r requirements.txt
for f in ['src/backbones/mobilenetv3.py',
          'configs/exp_phase5_mbnet_parallel_evidential.yaml',
          'scripts/step8_backbone_compare.py']:
    assert os.path.exists(f), f'MISSING {f} — push the Step 8 commit to this BRANCH.'
print('Step 8 code present — good.')

## 2. Stage data from the attached Kaggle Dataset (bpeft-data)

Resilient: accepts the CIFAR-100 tarball OR the (possibly nested) extracted folder.
Rule 5 above applies — the TinyImageNet **zip** is copied, never extracted.

In [ ]:
import os, shutil, hashlib, glob
os.makedirs('data/svhn', exist_ok=True)
CIFAR_MD5 = 'eb9058c3a382ffc7106e4002c42a8d85'

def _find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

def _find_cifar_root():
    for meta in glob.glob('/kaggle/input/**/meta', recursive=True):
        d = os.path.dirname(meta)
        if os.path.exists(os.path.join(d, 'train')) and os.path.exists(os.path.join(d, 'test')):
            return d
    return None

if os.path.isdir('data/cifar-100-python') or \
   (os.path.exists('data/cifar-100-python.tar.gz') and os.path.getsize('data/cifar-100-python.tar.gz') > 0):
    print('[cifar] already staged')
else:
    tar = _find_file('cifar-100-python.tar.gz'); root = _find_cifar_root()
    if tar:
        shutil.copy(tar, 'data/cifar-100-python.tar.gz')
        assert hashlib.md5(open('data/cifar-100-python.tar.gz','rb').read()).hexdigest() == CIFAR_MD5
        print('[cifar] staged tarball (md5 OK)')
    elif root:
        shutil.copytree(root, 'data/cifar-100-python')
        print(f'[cifar] copied extracted files from {root}')
    else:
        raise AssertionError('CIFAR-100 not found under /kaggle/input — attach bpeft-data (+ Add Input).')

dst = 'data/svhn/test_32x32.mat'
if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
    src = _find_file('test_32x32.mat'); assert src, 'test_32x32.mat not found under /kaggle/input'
    shutil.copy(src, dst); print(f'[svhn] staged ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print('[svhn] already present')

tin = _find_file('tiny-imagenet-200.zip')
if tin: shutil.copy(tin, 'data/tiny-imagenet-200.zip'); print('[tin] staged zip from dataset (NOT extracted)')
else:   print('[tin] not in dataset — downloads at runtime (Internet ON)')

## 3. Build the frozen CIFAR-FS split

Re-materialises the canonical Bertinetto 64/16/20 file (rule 1). The assert is
the guard against silently training on the synthetic fallback.

In [ ]:
!python scripts/build_cifar_fs_split.py
import json
sp = json.load(open('data/cifar_fs_split.json'))
print('split status:', sp.get('_status'), '| sizes:', {k: len(v) for k, v in sp.items() if isinstance(v, list)})
assert sp.get('_status') != 'synthetic_fallback', 'split is the SYNTHETIC fallback — fix before running'

## 4. Tests

`test_mobilenetv3.py` is the new Step 8 suite; its last three tests are
regression guards asserting ResNet-18's placement sites and param counts are
byte-for-byte what Step 6 reported (Step 8 generalised the code they share).

In [ ]:
!python -u -m pytest -q tests/test_mobilenetv3.py

In [ ]:
# Full suite (Steps 1-8). Expect 107 + the new Step 8 tests.
!python -u -m pytest -q

## 5. Backbone sanity check — feature dim, placement sites, param counts

Cheap, and it verifies the two things most likely to be wrong on a new backbone:
the 576-d pooled-feature contract, and *where* the in-block adapters actually
landed.

In [ ]:
import torch
from src.backbones import build_backbone, mobilenetv3_stage_paths, backbone_feature_dim
from src.adapters import build_adapter, infer_block_channels
from src.utils import count_trainable_params, load_config
from src.models import build_model

bb = build_backbone('mobilenetv3_small')
with torch.no_grad():
    feat = bb(torch.randn(2, 3, 224, 224))
print('pooled feature:', tuple(feat.shape), '| registry dim:', backbone_feature_dim('mobilenetv3_small'))
assert feat.shape[1] == 576

print('\nbackbone params (frozen):', f"{sum(p.numel() for p in bb.parameters()):,}")
print('placement sites:')
for p in mobilenetv3_stage_paths(bb):
    print(f'  {p:14s} {infer_block_channels(bb.get_submodule(p)):>4d} ch')

print('\ntrainable params per config:')
for name in ['exp_phase5_mbnet_bottleneck_evidential', 'exp_phase5_mbnet_bottleneck_softmax',
             'exp_phase5_mbnet_parallel_evidential',   'exp_phase5_mbnet_parallel_softmax']:
    cfg = load_config(f'configs/{name}.yaml')
    m = build_model(cfg)
    ad = m.adapter
    where = (', '.join(f'{p}({c}ch)' for p, c in zip(ad.stage_paths, ad.stage_channels))
             if hasattr(ad, 'stage_paths') else 'post_pool (576->16->576)')
    print(f'  {name:42s} {count_trainable_params(m):>8,d}   {where}')

## 6. Train + evaluate the 4 MobileNetV3-Small configs

600 test episodes each, with the full OOD list (`--use-tinyimagenet
--use-gaussian`) so the mbnet rows line up column-for-column with the ResNet-18
parallel row from Step 7.

Resumable: a config whose result JSON already exists is skipped, so a re-run
after a session timeout picks up where it stopped. The collapse guard
(`collapse_threshold: 0.25`) is left ON — if a new backbone's evidential run
degenerates it aborts after epoch 1 instead of burning the session.

In [ ]:
import subprocess, os, time
NUM_EPISODES = 600
RUNS = [
    # (config name, results-suffix)
    ('exp_phase5_mbnet_bottleneck_evidential', 'phase5_mbnet'),
    ('exp_phase5_mbnet_bottleneck_softmax',    'phase5_mbnet'),
    ('exp_phase5_mbnet_parallel_evidential',   'phase5_mbnet_parallel'),
    ('exp_phase5_mbnet_parallel_softmax',      'phase5_mbnet_parallel'),
]

def head_desc(name):
    return 'prototype-evidential' if name.endswith('evidential') else 'prototype-softmax'

def result_path(name, suffix):
    return f'results/{suffix}_bottleneck_{head_desc(name)}_metrics.json'

def run(cmd):
    print('>>>', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

status = {}
for name, suffix in RUNS:
    out = result_path(name, suffix)
    if os.path.exists(out):
        print(f'== SKIP {name} (found {out}) =='); status[name] = 'skip (exists)'; continue
    print(f'\n{"="*72}\n== {name} ==\n{"="*72}', flush=True)
    t0 = time.time()
    cfg = f'configs/{name}.yaml'
    rc = run(['python', '-u', 'scripts/train.py', '--config', cfg, '--wandb-mode', 'disabled'])
    if rc != 0:
        status[name] = f'TRAIN failed (rc={rc})'; continue
    rc = run(['python', '-u', 'scripts/evaluate.py', '--config', cfg,
              '--num-episodes', str(NUM_EPISODES), '--wandb-mode', 'disabled',
              '--results-suffix', suffix, '--use-tinyimagenet', '--use-gaussian'])
    ok = (rc == 0 and os.path.exists(out))
    status[name] = f'OK ({(time.time()-t0)/60:.0f} min)' if ok else f'EVAL failed (rc={rc})'

print('\nRUN STATUS')
for name, _s in RUNS:
    print(f'  {name:44s} {status.get(name, "not run")}')

## 7. Consolidate the cross-backbone table + plots

ResNet-18 rows are read from the already-committed Step 4.5 / Step 7 JSONs — no
re-run, so those numbers are exactly as previously reported.

In [ ]:
!python scripts/step8_backbone_compare.py

In [ ]:
import json
print(json.dumps(json.load(open('results/phase5_backbone_table.json')), indent=2, sort_keys=True))

In [ ]:
from IPython.display import Image, display
display(Image('results/step8_backbone_comparison.png'))
display(Image('results/step8_params_vs_accuracy.png'))

## 8. Download the models + results

Packs everything worth keeping into **one zip at `/kaggle/working`**:

* `checkpoints/*.pt` — the 4 trained adapters (**the models**)
* `results/phase5_mbnet*` + `phase5_backbone_table.json` + the 2 Step 8 PNGs
* `MANIFEST.txt` — sizes, sha256, git commit, so a downloaded bundle is traceable

Three ways to get it off Kaggle: the **link this cell prints**, the **Output**
panel on the right, or `Save Version` → the version's *Output* tab. Commit the
`results/*` files into the repo locally; checkpoints stay out of git (they are
gitignored — large binaries).

In [ ]:
import glob, hashlib, os, subprocess, zipfile
from pathlib import Path

REPO_DIR = '/kaggle/working/thesis'
os.chdir(REPO_DIR)
ZIP = '/kaggle/working/step8_mbnet_artifacts.zip'

MODELS  = sorted(glob.glob('checkpoints/model_phase2_mbnet_*.pt'))
RESULTS = (sorted(glob.glob('results/phase5_mbnet*'))
           + [p for p in ['results/phase5_backbone_table.json',
                          'results/step8_backbone_comparison.png',
                          'results/step8_params_vs_accuracy.png'] if os.path.exists(p)])

print(f'{len(MODELS)} model checkpoint(s), {len(RESULTS)} result file(s)')
if len(MODELS) < 4:
    print('[warn] fewer than 4 checkpoints — did every config train?')

def sha256(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(buf), b''):
            h.update(chunk)
    return h.hexdigest()

commit = subprocess.run(['git', 'log', '--oneline', '-1'],
                        capture_output=True, text=True).stdout.strip()
lines = [
    'Step 8 — MobileNetV3-Small backbone (Phase 5, RQ4 prep)',
    f'repo commit: {commit}',
    'protocol   : CIFAR-FS Bertinetto, 5-way 5-shot, 600 frozen test seeds (0..599)',
    'OOD pools  : svhn_far, gaussian_far, cifar100_near, tin_near (500 each)',
    '',
    'FILES (size / sha256):',
]
for p in MODELS + RESULTS:
    lines.append(f'  {p:70s} {os.path.getsize(p)/1e6:8.2f} MB  {sha256(p)[:16]}')
Path('MANIFEST.txt').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in MODELS + RESULTS + ['MANIFEST.txt']:
        z.write(p, arcname=p)
print(f'\nwrote {ZIP}  ({os.path.getsize(ZIP)/1e6:.1f} MB)')

# Clickable link (Kaggle serves /kaggle/working; FileLink needs a path relative to it).
os.chdir('/kaggle/working')
from IPython.display import FileLink, display
display(FileLink(os.path.basename(ZIP)))
os.chdir(REPO_DIR)

### Optional: download the files individually instead

Skip the zip and grab single files (handy if you only want the JSONs).

In [ ]:
import os, glob
from IPython.display import FileLink, display
os.chdir('/kaggle/working')
for p in (sorted(glob.glob('thesis/results/phase5_mbnet*'))
          + sorted(glob.glob('thesis/results/step8_*'))
          + ['thesis/results/phase5_backbone_table.json']
          + sorted(glob.glob('thesis/checkpoints/model_phase2_mbnet_*.pt'))):
    if os.path.exists(p):
        display(FileLink(p))
os.chdir('/kaggle/working/thesis')

## 9. After the run — what to write down

1. Transcribe the §7 table into `step_writeups/step8.txt` §4 (**copy the numbers,
   never retype from memory** — honesty rule, `thesis_implementation_instructions.txt` §6).
2. Note each run's `best_val_epoch` from the training log: an early stop at a
   sensible epoch is the evidence that accuracy is genuine convergence, not
   overfitting.
3. Tick Step 8's boxes in `progress.txt` and record the headline there.
4. Commit `results/phase5_mbnet*`, `results/phase5_backbone_table.json` and the
   two PNGs locally (a human commits — never the agent).